In [1]:

%load_ext autoreload 
%autoreload 2
from tqdm import tqdm
from omm.omm import ProteinImplicit
import mdtraj as md 
import numpy as np
import torch
import subprocess
from einops import rearrange, reduce, repeat
import os


In [2]:
parent_dir = "/home/alelic99/boltz-likelihoods/boltz_results_chignolin_inference/predictions/chignolin"
protein_name = "chignolin"



In [3]:
# load the diffusion structures as a concatenated pdb

diffusion_structures = []
file_list = os.listdir(parent_dir)
file_list = sorted(file_list, key=lambda x: int(x.split("_")[-1].split(".")[0]))
print(file_list)


['chignolin_model_0.pdb', 'chignolin_model_1.pdb', 'chignolin_model_2.pdb', 'chignolin_model_3.pdb', 'chignolin_model_4.pdb', 'chignolin_model_5.pdb', 'chignolin_model_6.pdb', 'chignolin_model_7.pdb', 'chignolin_model_8.pdb', 'chignolin_model_9.pdb']


In [4]:
# only run this once!

for filename in tqdm(file_list):
    if filename.endswith(".pdb"):
        diffusion_structures.append(md.load(f"{parent_dir}/{filename}"))
diffusion_structures = md.join(diffusion_structures)
diffusion_structures.save(f"{parent_dir}/all_diffusion_structures.pdb")

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 530.47it/s]


In [5]:
# otherwise just load the concatenated pdb

diffusion_structures = md.load(f"{parent_dir}/all_diffusion_structures.pdb")

In [6]:
def compute_force_from_traj(
    traj: md.Trajectory, 
    amber_filename:str, 
    num_relax_steps:int=0, 
    temperature:float=300
):
    """Compute the heavy atom forces at every trajectory frame.

    Parameters
    ----------
    traj : md.Trajectory
        Trajectory to compute forces for.
    amber_filename : str
        Name of the .prmtop and .(inp)crd files of the system.
    
    Notes
    -----
    Assumes kT units with T = 300 Kelvin unless otherwise specified.
    """    
    simulation_args = {
        "temperature": temperature, 
        "temperature_units": "kelvin", 
        "friction": 100.0, "dt": 0.00002, 
        "time_units": "picoseconds", 
        "prior_weight": None, 
        "integrator_to_use": "overdamped", 
        "do_energy_minimization": False, 
        "chk_freq": 10000000, 
        "device": "CPU", 
        "fix":"backbone"
    }
    solvent_args = {
        "implicit_solvent": "OBC2", 
        "implicit_solvent_kappa": 0.1, 
        "implicit_solvent_kappa_length_units": "nanometer"
    }

    p = ProteinImplicit(
        filename = amber_filename, chk=0,
        simulation_args=simulation_args,
        solvent_args=solvent_args,
        save_filename = f"./"
        )

    gen_positions = traj.xyz # mdtraj saves in nanometers by default
    final_forces = []

    for i, position in tqdm(enumerate(gen_positions), mininterval=10):
        # Need to somehow add hydrogens to each position tensor. 
        pos, pe, ke, forces = p.relax_energies(
            10 * position, # convert to angstroms
            velocities=True,
            num_relax_steps=num_relax_steps, # 0 relax steps means no relaxation, just get energies
            length_units="angstroms",
            time_units="picoseconds",
            energy_units="kilocalories_per_mole",
            # energy_units="kT"
        )
        final_forces.append(forces)

    return np.stack(final_forces, axis=0)

In [7]:
print(f"""
    source leaprc.ff99SBxildn
    protein = loadPDB {parent_dir}/{'chignolin_model_0.pdb'}
    check protein
    savePdb protein amber_{protein_name}_single.pdb
    saveAmberParm protein amber_{protein_name}_single.prmtop amber_{protein_name}_single.crd
    quit
    """)


    source leaprc.ff99SBxildn
    protein = loadPDB /home/alelic99/boltz-likelihoods/boltz_results_chignolin_inference/predictions/chignolin/chignolin_model_0.pdb
    check protein
    savePdb protein amber_chignolin_single.pdb
    saveAmberParm protein amber_chignolin_single.prmtop amber_chignolin_single.crd
    quit
    


In [8]:
diffusion_structures.n_atoms

92

In [9]:
import subprocess
import sys # Make sure to import sys

# ... (your tleap_input definition) ...
tleap_input = f"""
    source leaprc.ff99SBxildn
    protein = loadPDB {parent_dir}/{'chignolin_model_0.pdb'}
    check protein
    savePdb protein amber_{protein_name}_single.pdb
    saveAmberParm protein amber_{protein_name}_single.prmtop amber_{protein_name}_single.crd
    quit
    """

print(f"Running tleap on {parent_dir}/{file_list[0]}...")

try:
    result = subprocess.run(
        ["tleap", "-f", "-"],
        input=tleap_input,       # 'text=True' handles encoding
        capture_output=True,   # <-- Capture stdout and stderr
        text=True,             # <-- Decode output as text
        check=True             # <-- Keep this to catch the error
    )
    print("tleap (first call) succeeded.")
    print("STDOUT:", result.stdout)

except subprocess.CalledProcessError as e:
    print(f"--- TLEAP FAILED (Exit Code: {e.returncode}) ---", file=sys.stderr)
    print("\n--- TLEAP STDOUT ---", file=sys.stderr)
    print(e.stdout, file=sys.stderr)
    print("\n--- TLEAP STDERR ---", file=sys.stderr)
    print(e.stderr, file=sys.stderr) # <-- This will almost certainly contain the error

# Note: The script will still stop here because of the error, 
# but now you will know *why* it stopped.

Running tleap on /home/alelic99/boltz-likelihoods/boltz_results_chignolin_inference/predictions/chignolin/chignolin_model_0.pdb...
tleap (first call) succeeded.
STDOUT: -I: Adding /data2/scratch/group_scratch/amber/ambertools25/dat/leap/prep to search path.
-I: Adding /data2/scratch/group_scratch/amber/ambertools25/dat/leap/lib to search path.
-I: Adding /data2/scratch/group_scratch/amber/ambertools25/dat/leap/parm to search path.
-I: Adding /data2/scratch/group_scratch/amber/ambertools25/dat/leap/cmd to search path.
-f: Source -.

Welcome to LEaP!
(no leaprc in search path)
Sourcing: /data2/scratch/group_scratch/amber/ambertools25/dat/leap/cmd/leaprc
----- Source: /data2/scratch/group_scratch/amber/ambertools25/dat/leap/cmd/leaprc.ff99SBxildn
----- Source of /data2/scratch/group_scratch/amber/ambertools25/dat/leap/cmd/leaprc.ff99SBxildn done
Log file: ./leap.log
Loading parameters: /data2/scratch/group_scratch/amber/ambertools25/dat/leap/parm/parm99x.dat
Reading title:
PARM99 for DNA,

In [10]:
# run the amber files to add missing hydrogens and OXT and save prmtop and crd files

tleap_input = f"""
    source leaprc.ff99SBxildn
    protein = loadPDB {parent_dir}/{file_list[0]}
    check protein
    savePdb protein amber_{protein_name}_single.pdb
    saveAmberParm protein amber_{protein_name}_single.prmtop amber_{protein_name}_single.crd
    quit
    """

result = subprocess.run(
    ["tleap", "-f", "-"],  # "-" means read from stdin
    input=tleap_input.encode(),
    stdout=subprocess.DEVNULL,  # suppress normal output
    check=True
)

# diffusion iid inference traj
traj_for_forces_path = f"{parent_dir}/all_diffusion_structures.pdb"

tleap_input = f"""
    source leaprc.ff99SBxildn
    protein = loadPDB {traj_for_forces_path}
    saveAmberParm protein amber_{protein_name}_traj.prmtop amber_{protein_name}_traj.crd
    quit
    """

result = subprocess.run(
    ["tleap", "-f", "-"],  # "-" means read from stdin
    input=tleap_input.encode(),
    stdout=subprocess.DEVNULL,  # suppress normal output
    check=True
)

# turn the prmtop and crd files into a pdb file with hydrogens
subprocess.run(f"ambpdb -p amber_{protein_name}_traj.prmtop -c amber_{protein_name}_traj.crd > amber_{protein_name}_traj.pdb", shell=True, check=True)

CompletedProcess(args='ambpdb -p amber_chignolin_traj.prmtop -c amber_chignolin_traj.crd > amber_chignolin_traj.pdb', returncode=0)

In [11]:
# the fixed trajectory in amber format needs to be reshaped

amber_traj = md.load(f"amber_{protein_name}_traj.pdb")
amber_top = md.load(f"amber_{protein_name}_single.pdb").topology
print(f"{amber_traj.xyz.shape = }")

amber_pos = rearrange(amber_traj.xyz, "1 (frame atom) dim -> frame atom dim", 
                      frame=diffusion_structures.xyz.shape[0],)
fixed_amber_traj = md.Trajectory(amber_pos, amber_top).center_coordinates()
fixed_amber_traj.save(f"fixed_amber_{protein_name}_traj.pdb")
print(f"{fixed_amber_traj.xyz.shape = }")

amber_traj.xyz.shape = (1, 1660, 3)
fixed_amber_traj.xyz.shape = (10, 166, 3)


In [13]:
# finally, compute the forces

gt_forces = compute_force_from_traj(traj=fixed_amber_traj, amber_filename=f"amber_{protein_name}_single", num_relax_steps=0, temperature=300)
heavy_atom_indices = amber_top.select("not element H and not name OXT")
gt_forces = torch.tensor(gt_forces[:, heavy_atom_indices, :])
print(torch.mean(torch.norm(gt_forces, dim=-1)))


/home/alelic99/boltz-likelihoods/.venv/lib/python3.10/site-packages/openmm/app/internal/amber_file_parser.py:1168: UserWarning: Non-optimal GB parameters detected for GB model OBC2
  warnings.warn(


(166, 3)


10it [00:00, 140.71it/s]


166
166
166
166
166
166
166
166
166
166
tensor(1403651.1733, dtype=torch.float64)


In [14]:
gt_forces.shape

torch.Size([10, 92, 3])